# Portfolio Optimization

This notebook uses historical returns and predicted volatility to build and compare portfolio strategies.

The goal is to compare simple portfolio baselines against optimized portfolios. The strategies include an equal-weight portfolio, a historical minimum-volatility portfolio, a predictive minimum-volatility portfolio, and a maximum Sharpe ratio portfolio.

# Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from scipy.optimize import minimize

import matplotlib.pyplot as plt

# Paths

In [2]:
INTEGRATED_DATA_PATH = Path("../../data/processed/integrated")
MODELING_PATH = Path("../../data/processed/modeling")
PORTFOLIO_OUTPUT_PATH = Path("../../data/processed/portfolio_optimization")

In [4]:
if not PORTFOLIO_OUTPUT_PATH.exists():
    PORTFOLIO_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
else:
    print(f"Directory {PORTFOLIO_OUTPUT_PATH} already exists.") 

Directory ..\..\data\processed\portfolio_optimization already exists.


# Load Data

## Daily Market Data

Using daily market data for returns

In [5]:
daily_market_data = pd.read_csv(
    INTEGRATED_DATA_PATH / "daily_market_data.csv",
    parse_dates=["Date"]
)

## Random Forest

Using this one because it performed the best in the comparison file

In [6]:
rf_predictions = pd.read_csv(
    MODELING_PATH / "random_forest" / "test_predictions.csv",
    parse_dates=["Date"]
)

### Validate Loading Data Worked

In [8]:
daily_market_data.head()

,Date,ticker,adjusted_close,daily_return,risk_free_rate_pct,risk_free_rate_decimal,cpi_index,cpi_pct_change,company_name,gics_sector,asset_type
0,2018-01-02,AAPL,40.267082,NaN,1.44,0.0144,248.859,0.0,Apple Inc.,Information Technology,Stock
1,2018-01-03,AAPL,40.260067,-0.000174,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
2,2018-01-04,AAPL,40.447075,0.004645,1.41,0.0141,248.859,0.0,Apple Inc.,Information Technology,Stock
3,2018-01-05,AAPL,40.907566,0.011385,1.39,0.0139,248.859,0.0,Apple Inc.,Information Technology,Stock
4,2018-01-08,AAPL,40.755634,-0.003714,1.45,0.0145,248.859,0.0,Apple Inc.,Information Technology,Stock


In [10]:
daily_market_data.shape

(42210, 11)

In [9]:
rf_predictions.head()

,Date,ticker,future_volatility_20d,predicted_future_volatility_20d
0,2024-01-02,MSFT,0.010424,0.013266
1,2024-01-02,PG,0.011748,0.011599
2,2024-01-02,CAT,0.015127,0.013530
3,2024-01-02,AGG,0.003369,0.004461
4,2024-01-02,KO,0.006132,0.008131


In [11]:
rf_predictions.shape

(10101, 4)

# Create Return Matrix

In [12]:
returns_matrix = daily_market_data.pivot(
    index="Date",
    columns="ticker",
    values="daily_return"
).sort_index()